In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import shap
import pickle
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid")
pd.set_option('display.max_columns', None)

# Initialize SHAP's JavaScript visualization (needed for some plots)
shap.initjs()

print("All imports successful ✅")

In [ ]:
# ── Load data ────────────────────────────────────────────────
X_train = pd.read_csv('../data/X_train_final.csv')
X_test  = pd.read_csv('../data/X_test_final.csv')
y_test  = pd.read_csv('../data/y_test.csv').squeeze()

# ── Fix NaNs (same guard as Phase 5) ─────────────────────────
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(strategy='median')
X_train = pd.DataFrame(imputer.fit_transform(X_train), columns=X_train.columns)
X_test  = pd.DataFrame(imputer.transform(X_test),      columns=X_test.columns)

# ── Load best model and all models ───────────────────────────
with open('../models/best_model.pkl', 'rb') as f:
    best_model = pickle.load(f)

with open('../models/all_models.pkl', 'rb') as f:
    all_models = pickle.load(f)

print(f"Best model type: {type(best_model).__name__}")
print(f"X_test shape: {X_test.shape}")
print(f"NaNs in X_test: {X_test.isnull().sum().sum()} ✅")

## What is SHAP?

SHAP is based on game theory. It treats each feature as a "player"
contributing to the final prediction (the "payout").

The SHAP value for a feature tells us:
- **Positive SHAP value** → this feature pushed the prediction TOWARD attrition (leaving)
- **Negative SHAP value** → this feature pushed the prediction AWAY from attrition (staying)
- **Magnitude** → how strongly the feature influenced this particular prediction

### Why SHAP over feature importance?
- Regular feature importance says "OverTime is important" globally
- SHAP says "for THIS employee, OverTime pushed their risk up by 0.23"
- SHAP works at both the global level (all employees) AND individual level (one employee)
- SHAP is model-agnostic — works on any model the same way

In [ ]:
# TreeExplainer is optimized for tree-based models (RF, XGBoost)
# It is much faster than the generic KernelExplainer
explainer = shap.TreeExplainer(best_model)

# Compute SHAP values for the entire test set
# This tells us WHY the model made each prediction
print("Computing SHAP values... (this may take 30-60 seconds)")
shap_values = explainer.shap_values(X_test)

# Handle output shape differences between model types
model_type = type(best_model).__name__

if model_type == 'RandomForestClassifier':
    # Random Forest returns shap values for both classes → take class 1 (Left)
    shap_vals = shap_values[:, :, 1]
else:
    # XGBoost and others return a single array directly
    shap_vals = shap_values

print(f"Model type     : {model_type}")
print(f"SHAP values shape: {shap_vals.shape}")
print(f"  → {shap_vals.shape[0]} employees × {shap_vals.shape[1]} features ✅")

In [ ]:
plt.figure()
shap.summary_plot(
    shap_vals,
    X_test,
    plot_type="dot",
    max_display=20,
    show=False
)
plt.title("SHAP Summary Plot — Feature Impact on Attrition Prediction",
          fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('../data/shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: shap_summary.png ✅")

In [ ]:
plt.figure(figsize=(10, 7))
shap.summary_plot(
    shap_vals,
    X_test,
    plot_type="bar",
    max_display=20,
    show=False
)
plt.title("Mean |SHAP Value| — Average Feature Importance",
          fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('../data/shap_bar.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: shap_bar.png ✅")

In [ ]:
# Find the most important feature by mean absolute SHAP
top_feature = pd.Series(
    np.abs(shap_vals).mean(axis=0),
    index=X_test.columns
).idxmax()

second_feature = pd.Series(
    np.abs(shap_vals).mean(axis=0),
    index=X_test.columns
).nlargest(2).index[1]

print(f"Top feature: {top_feature}")
print(f"Second feature: {second_feature}")

plt.figure()
shap.dependence_plot(
    top_feature,
    shap_vals,
    X_test,
    interaction_index=second_feature,
    show=False
)
plt.title(f"SHAP Dependence: {top_feature} (colored by {second_feature})",
          fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/shap_dependence.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: shap_dependence.png ✅")

In [ ]:
# Find the employee the model was MOST confident would leave
leave_probabilities = best_model.predict_proba(X_test)[:, 1]
highest_risk_idx = np.argmax(leave_probabilities)
lowest_risk_idx  = np.argmin(leave_probabilities)

print(f"Highest risk employee — index: {highest_risk_idx}")
print(f"  Predicted probability of leaving: {leave_probabilities[highest_risk_idx]:.1%}")
print(f"  Actual outcome: {'Left ✅' if y_test.iloc[highest_risk_idx] == 1 else 'Stayed'}")

print(f"\nLowest risk employee — index: {lowest_risk_idx}")
print(f"  Predicted probability of leaving: {leave_probabilities[lowest_risk_idx]:.1%}")
print(f"  Actual outcome: {'Left' if y_test.iloc[lowest_risk_idx] == 1 else 'Stayed ✅'}")

In [ ]:
# Waterfall plot for the highest-risk employee
shap_explanation = shap.Explanation(
    values     = shap_vals[highest_risk_idx],
    base_values= explainer.expected_value if model_type != 'RandomForestClassifier'
                 else explainer.expected_value[1],
    data       = X_test.iloc[highest_risk_idx].values,
    feature_names = X_test.columns.tolist()
)

plt.figure()
shap.waterfall_plot(shap_explanation, max_display=15, show=False)
plt.title(f"Why did the model predict this employee would LEAVE?\n"
          f"(Predicted probability: {leave_probabilities[highest_risk_idx]:.1%})",
          fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/shap_waterfall_high_risk.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: shap_waterfall_high_risk.png ✅")

In [ ]:
shap_explanation_low = shap.Explanation(
    values     = shap_vals[lowest_risk_idx],
    base_values= explainer.expected_value if model_type != 'RandomForestClassifier'
                 else explainer.expected_value[1],
    data       = X_test.iloc[lowest_risk_idx].values,
    feature_names = X_test.columns.tolist()
)

plt.figure()
shap.waterfall_plot(shap_explanation_low, max_display=15, show=False)
plt.title(f"Why did the model predict this employee would STAY?\n"
          f"(Predicted probability of leaving: {leave_probabilities[lowest_risk_idx]:.1%})",
          fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/shap_waterfall_low_risk.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: shap_waterfall_low_risk.png ✅")

In [ ]:
# Take a representative sample for the heatmap (full test set can be slow)
sample_size = min(100, len(X_test))
sample_idx  = np.random.choice(len(X_test), sample_size, replace=False)

shap_explanation_full = shap.Explanation(
    values        = shap_vals[sample_idx],
    base_values   = np.full(sample_size,
                            explainer.expected_value if model_type != 'RandomForestClassifier'
                            else explainer.expected_value[1]),
    data          = X_test.iloc[sample_idx].values,
    feature_names = X_test.columns.tolist()
)

plt.figure()
shap.plots.heatmap(shap_explanation_full, max_display=15, show=False)
plt.title("SHAP Heatmap — Feature Contributions Across All Sampled Employees",
          fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/shap_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: shap_heatmap.png ✅")

In [ ]:
# Build a clean summary table of SHAP importance
shap_importance = pd.DataFrame({
    'Feature'         : X_test.columns,
    'Mean_SHAP'       : np.abs(shap_vals).mean(axis=0),
    'Positive_Impact' : (shap_vals > 0).mean(axis=0),   # % of employees where feature increased risk
    'Negative_Impact' : (shap_vals < 0).mean(axis=0),   # % of employees where feature decreased risk
}).sort_values('Mean_SHAP', ascending=False).reset_index(drop=True)

shap_importance['Rank'] = shap_importance.index + 1

print("Top 15 Features by SHAP Importance:")
print(shap_importance.head(15).to_string(index=False))

# Save
shap_importance.to_csv('../data/shap_importance.csv', index=False)
print("\nSaved: shap_importance.csv ✅")

In [ ]:
# Save shap values and explainer — needed for the Streamlit app
np.save('../models/shap_values.npy', shap_vals)

with open('../models/shap_explainer.pkl', 'wb') as f:
    pickle.dump(explainer, f)

print("Saved:")
print("  ✅ models/shap_values.npy")
print("  ✅ models/shap_explainer.pkl")

## SHAP Interpretability Summary

### Plots Generated
| Plot | What It Shows |
|---|---|
| Summary (Beeswarm) | Global feature importance + direction of effect per employee |
| Bar Chart | Clean ranking of top features by average SHAP impact |
| Dependence Plot | How the top feature's effect varies + interaction with 2nd feature |
| Waterfall (High Risk) | Why the model flagged the highest-risk employee |
| Waterfall (Low Risk) | Why the model predicted lowest-risk employee would stay |
| Heatmap | SHAP values across all sampled employees at once |

### Key Findings
- Features with highest mean |SHAP|: (fill in your top 3 from the bar chart)
- OverTime = Yes consistently shows the largest positive SHAP values
- Low MonthlyIncome/IncomePerYearExp pushes predictions strongly toward leaving
- High SatisfactionScore acts as a strong protective factor against attrition
- Individual explanations confirm the model reasons in a business-logical way